In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
train = pd.read_parquet("../data/interim/features_train.parquet")

In [ ]:
test = pd.read_parquet("../data/interim/features_test.parquet")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

In [4]:
print(train['trans_date_trans_time'].dtype)

datetime64[us]


In [5]:
#Row count
print("Train:", len(train))
print("Test:", len(test))


Train: 1296675
Test: 555719


In [6]:
#Sort train by time 
train = train.sort_values(
    "trans_date_trans_time"
).reset_index(drop=True)
#Putting transcations in a timeline

In [7]:
#Setting a cut date ( 21 March 2020)
cut_date = pd.to_datetime("2020-03-21 00:00:00")
#This means:
# Everything before this moment → train
# Everything at or after this moment → validation

In [8]:
new_train = train[
    train['trans_date_trans_time'] < cut_date # Strictly before the cut date
].copy()

In [9]:
#Creating a validation 
validation = train[
    train["trans_date_trans_time"] >= cut_date
].copy()
#On or after March 21.

In [10]:
test = test.copy()

In [11]:
print("New train:", len(new_train))
print("Validation:", len(validation))
print("Original train:", len(train))

print(
    "Do they add up?",
    len(new_train) + len(validation) == len(train)
)


New train: 1070966
Validation: 225709
Original train: 1296675
Do they add up? True


In [12]:
print("TRAIN")
print("Start:", new_train["trans_date_trans_time"].min())
print("End:", new_train["trans_date_trans_time"].max())

print("\nVALIDATION")
print("Start:", validation["trans_date_trans_time"].min())
print("End:", validation["trans_date_trans_time"].max())

print("\nTEST")
print("Start:", test["trans_date_trans_time"].min())
print("End:", test["trans_date_trans_time"].max())


TRAIN
Start: 2019-01-01 00:00:18
End: 2020-03-20 23:58:44

VALIDATION
Start: 2020-03-21 00:00:20
End: 2020-06-21 12:13:37

TEST
Start: 2020-06-21 12:14:25
End: 2020-12-31 23:59:34


In [13]:
#Calculate fraud rate
new_train["is_fraud"].mean()*100

np.float64(0.5835852865543817)

In [14]:
# final table:
def get_summary(data):
    return {
        "Start date": data["trans_date_trans_time"].min(),
        "End date": data["trans_date_trans_time"].max(),
        "Rows": len(data),
        "Fraud rate": data["is_fraud"].mean() * 100
    }

In [15]:
summary = pd.DataFrame({
    "Train": get_summary(new_train),
    "Validation": get_summary(validation),
    "Test": get_summary(test)
}).T

print(summary)


                     Start date             End date     Rows Fraud rate
Train       2019-01-01 00:00:18  2020-03-20 23:58:44  1070966   0.583585
Validation  2020-03-21 00:00:20  2020-06-21 12:13:37   225709   0.556469
Test        2020-06-21 12:14:25  2020-12-31 23:59:34   555719   0.385986


In [16]:
summary["Fraud rate"] = summary["Fraud rate"].round(3)


In [17]:
summary["Fraud rate"]

Train         0.584
Validation    0.556
Test          0.386
Name: Fraud rate, dtype: object

## Save the three sets

**Cut date: `2020-03-21`** (train = before, validation = on or after). This later moves into `params.yaml`, so it's never hidden inside code.

The split is saved once, and every later step reads these files. That way every notebook uses exactly the same sets, and nobody redoes the cut with a different date or forgets to sort.

- Input: `data/interim/features_train.parquet`, `features_test.parquet` (from 02)
- Output: `data/processed/train.parquet`, `val.parquet`, `test.parquet`

In [18]:
new_train.to_parquet("../data/processed/train.parquet", index=False)
validation.to_parquet("../data/processed/val.parquet", index=False)
test.to_parquet("../data/processed/test.parquet", index=False)

In [19]:
# Re-open the saved files to prove they match the table above
for name in ["train", "val", "test"]:
    saved = pd.read_parquet(f"../data/processed/{name}.parquet")
    print(f"{name}: {len(saved):,} rows, fraud rate {saved['is_fraud'].mean() * 100:.3f}%")

train: 1,070,966 rows, fraud rate 0.584%
val: 225,709 rows, fraud rate 0.556%
test: 555,719 rows, fraud rate 0.386%
